# 10 — Phase 1 DTW alignment (v1: typewell-only correlation)

First-cut implementation of the multi-reference alignment described in `docs/roadmap.md` Phase 1. This version aligns each toe-row's GR window only against the **typewell** GR series. The follow-on (lateral-heel as a second reference, fused with typewell) lives in a later commit.

Strategy:
1. For each train well, treat `TVT_input` (already NaN over the toe) as the eval mask — this is the same shape we'll see at test time.
2. For each NaN row, slide a window of lateral GR across the resampled typewell GR and pick the typewell TVT with the best Pearson correlation.
3. Compute RMSE against the ground-truth `TVT` column (available only in train) and compare to the carry-forward floor.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd

repo = Path.cwd().resolve()
while not (repo / 'pyproject.toml').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'src'))
from rogii.features.correlation import (
    resample_to_step, predict_tvt_via_correlation, rmse,
)
RAW = repo / 'data' / 'raw'
TRAIN = RAW / 'train'
print('Repo:', repo)
print('Train wells:', len(list(TRAIN.glob('*__horizontal_well.csv'))))

In [ ]:
wells = sorted({p.name.split('__')[0] for p in TRAIN.glob('*__horizontal_well.csv')})
print(f'{len(wells)} train wells')

def load(well: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    h = pd.read_csv(TRAIN / f'{well}__horizontal_well.csv')
    t = pd.read_csv(TRAIN / f'{well}__typewell.csv')
    return h, t

In [ ]:
def carry_forward_pred(h: pd.DataFrame) -> np.ndarray:
    """Carry-forward baseline matching notebooks/00. Returns predictions over the full lateral."""
    last_known = h['TVT_input'].dropna().iloc[-1] if h['TVT_input'].notna().any() else 0.0
    pred = h['TVT_input'].copy()
    pred[pred.isna()] = last_known
    return pred.to_numpy()

In [ ]:
def correlation_pred(h: pd.DataFrame, t: pd.DataFrame, *, window_size: int = 51) -> np.ndarray:
    md = h['MD'].to_numpy(float)
    gr = h['GR'].to_numpy(float)
    eval_mask = h['TVT_input'].isna().to_numpy()
    
    # Resample typewell GR onto a 1-ft TVT grid to match lateral MD step.
    tvt_grid, gr_grid = resample_to_step(
        t['TVT'].to_numpy(float), t['GR'].to_numpy(float), step=1.0,
    )
    
    # NaN-fill GR before correlation (imputation: nearest valid neighbor)
    if np.isnan(gr).any():
        gr_series = pd.Series(gr).ffill().bfill()
        gr = gr_series.to_numpy()
    
    last_known_tvt = h['TVT_input'].dropna().iloc[-1] if h['TVT_input'].notna().any() else float(tvt_grid[len(tvt_grid)//2])
    
    pred = predict_tvt_via_correlation(
        lateral_md=md, lateral_gr=gr, eval_mask=eval_mask,
        ref_depth=tvt_grid, ref_gr=gr_grid,
        window_size=window_size, last_known_tvt=last_known_tvt,
        drift_per_ft=0.5,
    )
    # Stitch in the known heel TVT_input values
    out = h['TVT_input'].to_numpy(float).copy()
    out[eval_mask] = pred[eval_mask]
    return out

In [ ]:
import time
rng = np.random.default_rng(0)
sample_wells = list(rng.choice(wells, size=10, replace=False))
print('Wells sampled:', sample_wells)

rows = []
t0 = time.time()
for well in sample_wells:
    h, t = load(well)
    eval_mask = h['TVT_input'].isna().to_numpy()
    y_true = h['TVT'].to_numpy(float)
    
    cf = carry_forward_pred(h)
    co = correlation_pred(h, t)
    
    rows.append({
        'well': well,
        'eval_rows': int(eval_mask.sum()),
        'rmse_carry_forward': rmse(y_true[eval_mask], cf[eval_mask]),
        'rmse_correlation': rmse(y_true[eval_mask], co[eval_mask]),
    })
elapsed = time.time() - t0
df = pd.DataFrame(rows)
df['delta'] = df['rmse_carry_forward'] - df['rmse_correlation']
print(df.to_string(index=False))
print()
print(f'aggregate carry-forward RMSE: {df.rmse_carry_forward.mean():.2f}')
print(f'aggregate correlation  RMSE: {df.rmse_correlation.mean():.2f}')
print(f'mean improvement (lower is better): {df.delta.mean():+.2f}')
print(f'wells where correlation beats carry-forward: {(df.delta > 0).sum()}/{len(df)}')
print(f'elapsed: {elapsed:.1f}s ({elapsed/len(sample_wells):.1f}s/well)')

## What's missing in v1

1. **Heel-as-reference.** The task brief Slide 9 says the lateral heel's GR has higher resolution than the typewell. v2 should fuse heel-DTW with typewell-DTW.
2. **Monotonicity within a regime.** Currently each row is matched independently; predictions may oscillate. Smoothing or regime-aware DTW would fix this.
3. **Trajectory regularization.** TVT changes should be consistent with `dZ/dMD`. Use Z to bound the TVT search.
4. **Offset-well prior** (Phase 2 in the roadmap).